In [ ]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [ ]:
# highest level of qualification by sex
# To download original dataset go to - 
# https://fingertips.phe.org.uk/
life_expectancy_csv = r"N:\Geodatabase\Raw_Data\OHID Health\OHID_Life_Expectancy.csv"
life_expectancy_average_csv = r"N:\Geodatabase\Raw_Data\OHID Health\OHID_Mortality_Rate.csv"

## 2. Input Link to MSOA 2011 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::middle-layer-super-output-areas-december-2011-boundaries-ew-bfe-v3/about

In [ ]:
msoa2011_shapefile = r"N:\Geodatabase\Raw_Data\Census 2021\Boundaries\Middle_Layer_Super_Output_Areas_Dec_2011_Boundaries_Full_Extent_BFE_EW_V3_2022_46367968394682200\MSOA_2011_EW_BFE_V3.shp"

## 3. National Benchmark threshold (%)

In [ ]:
x = 0.05 #5%

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [ ]:
life_expectancy_raw_data_df = pd.read_csv(life_expectancy_csv)

# STEP 1: FILTER FOR ROWS WHERE 'AREA TYPE' EQUALS 'MSOA' and 'TIME PERIOD' EQUALS '2021/22 - 23/24'
life_expectancy_df = life_expectancy_raw_data_df[
    (life_expectancy_raw_data_df['Area Type'] == 'MSOA') &
    (life_expectancy_raw_data_df['Time period'] == '2016 - 20')
]

# STEP 2: KEEP ONLY THE SPECIFIED COLUMNS
columns_to_keep = [
    'Area Code', 'Sex', 'Value'
]

life_expectancy_df = life_expectancy_df[columns_to_keep]

# STEP 3: RENAME THE SPECIFIED COLUMNS
life_expectancy_df.rename(columns={
    'Sex': 'sex',
    'Area Code': 'msoa11cd',    
}, inplace=True)

# STEP 4: CREATE SEPARATE DATAFRAMES FOR EACH UNIQUE VALUE IN 'INDICATOR NAME'
unique_sex = life_expectancy_df['sex'].unique()
dataframes = {}

for sex in unique_sex:
    dataframes[sex] = life_expectancy_df[life_expectancy_df['sex'] == sex]

#STEP 5: CREATE INDIVIDUAL DATAFRAME FROM DICTIONARY
male_life_expectancy_df = dataframes['Male']
female_life_expectancy_df = dataframes['Female']

#STEP 6: RENAME VALUE, COUNT AND DENOMINATOR FIELDS
# Define renaming mappings for each DataFrame
rename_mappings = {
    "male_life_expectancy_df": {
        'Value': 'male_life_expectancy',        
    },
    "female_life_expectancy_df": {
        'Value': 'female_life_expectancy',        
    }
}

# Iterate over DataFrames and apply renaming
for df_name, renames in rename_mappings.items():
    locals()[df_name].rename(columns=renames, inplace=True)

# STEP 7: MERGE DATAFRAMES
# drop fields to avoid duplication

# Drop the columns from the dataframe
male_life_expectancy_df = male_life_expectancy_df.drop(columns=['sex'], errors='ignore')
female_life_expectancy_df = female_life_expectancy_df.drop(columns=['sex'], errors='ignore')

#merge dataframes
life_expectancy_df = pd.merge(male_life_expectancy_df,
                              female_life_expectancy_df, 
                              how = 'left', 
                              on = 'msoa11cd'
)

# Display the updated dataframe
life_expectancy_df.head()

In [ ]:
life_expectancy_average_raw_data_df = pd.read_csv(life_expectancy_average_csv)

# STEP 1: FILTER FOR ROWS WHERE 'AREA TYPE' EQUALS 'MSOA' and 'TIME PERIOD' EQUALS '2016 - 20'
life_expectancy_average_df = life_expectancy_average_raw_data_df[
    (life_expectancy_average_raw_data_df['Area Type'] == 'MSOA') &
    (life_expectancy_average_raw_data_df['Time period'] == '2016 - 20')
]

# STEP 2: KEEP ONLY THE SPECIFIED COLUMNS
columns_to_keep = [
    'Indicator Name', 'Area Code', 'Value'
]

life_expectancy_average_df = life_expectancy_average_df[columns_to_keep]

# STEP 3: RENAME THE SPECIFIED COLUMNS
life_expectancy_average_df.rename(columns={
    'Indicator Name': 'indicator_name',
    'Area Code': 'msoa11cd',    
}, inplace=True)

# STEP 4: CREATE SEPARATE DATAFRAMES FOR EACH UNIQUE VALUE IN 'INDICATOR NAME'
unique_indicators = life_expectancy_average_df['indicator_name'].unique()
dataframes = {}

for indicator in unique_indicators:
    dataframes[indicator] = life_expectancy_average_df[life_expectancy_average_df['indicator_name'] == indicator]

#STEP 5: CREATE INDIVIDUAL DATAFRAME FROM DICTIONARY

life_expectancy_at_birth_df = dataframes['Life expectancy at birth, upper age band 90 and over']

#STEP 6: RENAME VALUE, COUNT AND DENOMINATOR FIELDS
# Define renaming mappings for each DataFrame
rename_mappings = {
    "life_expectancy_at_birth_df": {
        'Value': 'avg_life_expectancy'        
    }
}

# Iterate over DataFrames and apply renaming
for df_name, renames in rename_mappings.items():
    locals()[df_name].rename(columns=renames, inplace=True)

# STEP 7: MERGE DATAFRAMES
# drop fields to avoid duplication
# list of columns to drop
columns_to_drop = [
    'indicator_name', 
]

# Drop the columns from the dataframe
life_expectancy_at_birth_df = life_expectancy_at_birth_df.drop(columns=['indicator_name'], errors='ignore')

#merge dataframes

life_expectancy_df = pd.merge(life_expectancy_df,
                                      life_expectancy_at_birth_df, 
                                      how = 'left', 
                                      on = 'msoa11cd'
)


# Display the updated dataframe
life_expectancy_df.head()

## 4. Load GIS shapefile 

In [ ]:
# Attempt to read from the server first, fallback to local path if it fails
msoa2011boundary_df = gpd.read_file(msoa2011_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
msoa2011boundary_df.head()

## 5. GIS data management

### Remove Rename fields

In [ ]:
#Drop and rename fields
msoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)
msoa2011boundary_df.rename(columns = {'MSOA11CD':'msoa11cd','MSOA11NM':'msoa11nm'}, inplace = True)
msoa2011boundary_df.head()

### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [ ]:
#MSOA11 LOOKUP
msoa11_lookup_df = pd.read_excel(open(r"N:\Geodatabase\Raw_Data\Look up tables\MSOA_(2011)_to_MSOA_(2021)_to_Local_Authority_District_(2022)_Lookup_for_England_and_Wales_7815823766405070629(3).xlsx", 'rb'), sheet_name='MSOA_(2011)_to_MSOA_(2021)__0') 

columns_to_keep = [
    'MSOA11CD','LAD22CD','LAD22NM'
]

#drop unwanted fields
msoa11_lookup_cleaned_df = msoa11_lookup_df[columns_to_keep]

#rename msoa11cd to make merging cleaner
msoa11_lookup_cleaned_df.rename(columns={
    'MSOA11CD': 'msoa11cd',
    'LAD22CD': 'lad22cd',
    'LAD22NM': 'lad22nm',
}, inplace=True)

msoa11_lookup_cleaned_df.head()

In [ ]:
msoa2011boundary_df = pd.merge(msoa2011boundary_df, msoa11_lookup_cleaned_df, how = 'left', on = 'msoa11cd')
msoa2011boundary_df.head()

#### LAD22 to REGION22

In [ ]:
# Define the file path for lad to region
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

In [ ]:
msoa2011boundary_df = pd.merge(msoa2011boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
msoa2011boundary_df.head()

### Add data management fields

In [ ]:
# Add new data management fields with specified values
msoa2011boundary_df['data_source'] = "Office for Health Improvement & Disparities"
msoa2011boundary_df['data_resolution'] = "MSOA 2011"
msoa2011boundary_df['data_time_period'] = "2016 - 20"
msoa2011boundary_df['data_web_link'] = "https://fingertips.phe.org.uk/"
msoa2011boundary_df.head()

### Add 2021 Population

In [ ]:
# Define the file path for census 2021 population
file_path = r"N:\Geodatabase\Raw_Data\Census 2021\Population\MSOA2011_Population_Census_2021.csv"

# Read the Excel file
census_2021_pop_df = pd.read_csv(file_path)  # Reads the first sheet by default

# Display the first few rows
census_2021_pop_df.head()

In [ ]:
msoa2011boundary_df = pd.merge(msoa2011boundary_df, census_2021_pop_df, how = 'left', on = 'msoa11cd')
msoa2011boundary_df.head()

### Update Area

In [ ]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in msoa2011boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    msoa2011boundary_df['area_ha'] = msoa2011boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in msoa2011boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
msoa2011boundary_df.head()

# 7. Combine Geometry and data

In [ ]:
ohid_life_expectancy_msoa2011_gdb_df = pd.merge(msoa2011boundary_df, life_expectancy_df, how = 'left', on='msoa11cd')
ohid_life_expectancy_msoa2011_gdb_df.head()

In [ ]:
ohid_life_expectancy_msoa2011_gdb_df['avg_life_expectancy'] = ((ohid_life_expectancy_msoa2011_gdb_df['male_life_expectancy']*ohid_life_expectancy_msoa2011_gdb_df['male_population'])+(ohid_life_expectancy_msoa2011_gdb_df['female_life_expectancy']*ohid_life_expectancy_msoa2011_gdb_df['female_population']))/(ohid_life_expectancy_msoa2011_gdb_df['male_population']+ohid_life_expectancy_msoa2011_gdb_df['female_population'])
ohid_life_expectancy_msoa2011_gdb_df.head()

In [ ]:
# National average values
national_avg_male = 79.465500789683
national_avg_female = 83.1468255712653
national_avg = 81.36622

# Calculate tolerance ranges
tolerance_male = national_avg_male * x
tolerance_female = national_avg_female * x
tolerance_combined = national_avg_female * x

# Create the 'national_benchmark_male' column
ohid_life_expectancy_msoa2011_gdb_df['national_benchmark_male'] = ohid_life_expectancy_msoa2011_gdb_df['male_life_expectancy'].apply(
    lambda le: (
        'Above National Average' if le > national_avg_male + tolerance_male else
        'Below National Average' if le < national_avg_male - tolerance_male else
        'Similar to National Average'
    )
)

# Create the 'national_benchmark_female' column
ohid_life_expectancy_msoa2011_gdb_df['national_benchmark_female'] = ohid_life_expectancy_msoa2011_gdb_df['female_life_expectancy'].apply(
    lambda le: (
        'Above National Average' if le > national_avg_female + tolerance_female else
        'Below National Average' if le < national_avg_female - tolerance_female else
        'Similar to National Average'
    )
)

# Create the 'national_benchmark_female' column
ohid_life_expectancy_msoa2011_gdb_df['national_benchmark_combined'] = ohid_life_expectancy_msoa2011_gdb_df['avg_life_expectancy'].apply(
    lambda le: (
        'Above National Average' if le > national_avg + tolerance_combined else
        'Below National Average' if le < national_avg - tolerance_combined else
        'Similar to National Average'
    )
)

ohid_life_expectancy_msoa2011_gdb_df.head()

In [ ]:
ohid_life_expectancy_msoa2011_gdb_df = ohid_life_expectancy_msoa2011_gdb_df.drop_duplicates(subset='msoa11cd', keep='first')

# 9. Upload to geodatabase

In [ ]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "ohid_masoa2011_life_expectancy"  # Desired table name
primary_key_column = "msoa11cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [ ]:
# Ensure the GeoDataFrame has a valid CRS before writing
if ohid_life_expectancy_msoa2011_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    ohid_life_expectancy_msoa2011_gdb_df.set_crs(epsg=27700, inplace=True)

In [ ]:
# Publish the GeoDataFrame to PostGIS
ohid_life_expectancy_msoa2011_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")